In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 8
fig_height = 5
fig_format = 'retina'
fig_dpi = 96
interactivity = 'all'
is_shiny = False
is_dashboard = True
plotly_connected = False

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'QzpcVXNlcnNcam9oYW5cRG93bmxvYWRzXGRlZmkyLWRhc2hib2FyZFxkZWZpMi1kYXNoYm9hcmQ='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"C:\\Users\\johan\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\importlib\\_bootstrap.py": 1744135012.0, "C:\\Users\\johan\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\importlib\\_bootstrap_external.py": 1744135012.0, "C:\\Users\\johan\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\zipimport.py": 1744135014.0, "C:\\Users\\johan\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\codecs.py": 1744135012.0, "C:\\Users\\johan\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\encodings\\aliases.py": 1744135012.0, "C:\\Users\\johan\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\encodings\\__init__.py": 1744135012.0, "C:\\Users\\johan\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\encodings\\utf_8.py": 1744135012.0, "C:\\Users\\johan\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\abc.py": 1744135012.0, "C:\\Users\\johan\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\io.py": 1744135012.0, "C:\\Users\\johan\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\stat.

In [2]:
#| include: false
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from FinStoch import VasicekModel, MertonJumpDiffusion, OrnsteinUhlenbeck

px = pd.read_parquet("data/prices.parquet")
tickers = list(px.columns)

def selector(fig, labels, k, title):
    n = len(labels)
    for j, t in enumerate(fig.data):
        t.visible = j // k == 0
    buttons = [
        dict(label=l, method="update", args=[{"visible": [j // k == i for j in range(n * k)]}])
        for i, l in enumerate(labels)
    ]
    ranges = [dict(count=m, label=f"{m}m", step="month", stepmode="backward") for m in (1, 3, 6)]
    ranges += [dict(count=1, label="1y", step="year", stepmode="backward"), dict(step="all")]
    return fig.update_layout(
        title=title,
        updatemenus=[dict(buttons=buttons, x=0, y=1.18, xanchor="left")],
        xaxis=dict(rangeselector=dict(buttons=ranges), rangeslider=dict(visible=True)),
        yaxis_title="USD",
        legend=dict(orientation="h", y=-0.3),
    )

def montecarlo(model, name, rename, n=200):
    fig, rows = go.Figure(), {}
    for t in tickers:
        s = px[t].dropna()
        p = model.calibrate(s.values, dt=1 / 252)
        m = model(S0=s.iloc[0], **p, num_paths=n,
                  start_date=str(s.index[0].date()), end_date=str(s.index[-1].date()), granularity="D")
        sims = m.to_dataframe(m.simulate(seed=42))
        fig.add_scatter(x=s.index, y=s.values, name="Real", line=dict(color="black"))
        fig.add_scatter(x=sims.columns, y=sims.mean(), name=f"Media {name}")
        fig.add_scatter(x=sims.columns, y=sims.quantile(0.95), name="P95", line=dict(width=0), showlegend=False)
        fig.add_scatter(x=sims.columns, y=sims.quantile(0.05), name="P5–P95", line=dict(width=0),
                        fill="tonexty", fillcolor="rgba(128,128,128,0.3)")
        rows[t] = p
    return selector(fig, tickers, 4, f"Precio real vs. Monte Carlo {name} ({n} trayectorias)"), \
        pd.DataFrame(rows).T.rename(columns=rename)

In [3]:
fig = go.Figure()
for t in tickers:
    fig.add_scatter(x=px.index, y=px[t], name=t)
selector(fig, tickers, 1, "Precio de cierre ajustado")

In [4]:
fig, params = montecarlo(VasicekModel, "Vasicek", {"a": "κ", "mu": "θ", "sigma": "σ"})
fig

C:\Users\johan\AppData\Local\Temp\ipykernel_8136\1283740193.py:31: UserWarning: Estimated AR(1) coefficient phi=1.0006 is outside (0, 1). Data may not be mean-reverting. Clamping to valid range.
  p = model.calibrate(s.values, dt=1 / 252)
C:\Users\johan\AppData\Local\Temp\ipykernel_8136\1283740193.py:31: UserWarning: Estimated AR(1) coefficient phi=1.0012 is outside (0, 1). Data may not be mean-reverting. Clamping to valid range.
  p = model.calibrate(s.values, dt=1 / 252)


In [5]:
params.round(4)

,θ,σ,κ
AAPL,261.8733,42.0032,0.3196
AMZN,172.7493,49.7811,0.9583
GOOG,178.3526,36.6569,0.4094
JNJ,143.3830,25.9736,2.7737
JPM,90023.7674,37.4837,0.0003
META,20892.6822,120.6979,0.0036
MSFT,402.3642,75.7888,0.4521
NVDA,101854.2824,26.8731,0.0003
TSLA,267.4456,140.5200,1.3923
V,257.8675,54.4399,0.7018


In [6]:
fig, params = montecarlo(MertonJumpDiffusion, "Merton",
                         {"sigma": "σ", "mu": "μ", "lambda_j": "λ", "mu_j": "μ_J", "sigma_j": "σ_J"})
fig

In [7]:
params.round(4)

,μ,σ,λ,μ_J,σ_J
AAPL,0.3005,0.2386,24.6723,-0.0035,0.0443
AMZN,0.2368,0.2765,27.9395,-0.0026,0.0468
GOOG,0.2628,0.2259,42.6755,-0.0038,0.0381
JNJ,0.0460,0.1357,21.8919,0.0002,0.0314
JPM,0.1903,0.2079,28.9570,-0.0013,0.0481
META,0.3124,0.3252,14.3237,-0.0049,0.0860
MSFT,0.2514,0.2275,25.1526,-0.0040,0.0428
NVDA,0.7728,0.4647,12.4794,0.0060,0.0803
TSLA,0.7725,0.4918,37.1466,0.0003,0.0808
V,0.1489,0.1713,38.6891,-0.0040,0.0367


In [8]:
fig, params = montecarlo(OrnsteinUhlenbeck, "OU", {"theta": "θ", "mu": "μ", "sigma": "σ"})
fig

C:\Users\johan\AppData\Local\Temp\ipykernel_8136\1283740193.py:31: UserWarning: Estimated AR(1) coefficient phi=1.0006 is outside (0, 1). Data may not be mean-reverting. Clamping to valid range.
  p = model.calibrate(s.values, dt=1 / 252)
C:\Users\johan\AppData\Local\Temp\ipykernel_8136\1283740193.py:31: UserWarning: Estimated AR(1) coefficient phi=1.0012 is outside (0, 1). Data may not be mean-reverting. Clamping to valid range.
  p = model.calibrate(s.values, dt=1 / 252)


In [9]:
params.round(4)

,μ,σ,θ
AAPL,261.8733,42.0032,0.3196
AMZN,172.7493,49.7811,0.9583
GOOG,178.3526,36.6569,0.4094
JNJ,143.3830,25.9736,2.7737
JPM,90023.7674,37.4837,0.0003
META,20892.6822,120.6979,0.0036
MSFT,402.3642,75.7888,0.4521
NVDA,101854.2824,26.8731,0.0003
TSLA,267.4456,140.5200,1.3923
V,257.8675,54.4399,0.7018
